# Module 02: Truth Table & Boolean Simplification
**AE2.021 · USTH**

This notebook auto-generates the truth table for any Boolean expression and uses `sympy` to simplify
it, cross-checking against the manual simplification steps (De Morgan, absorption...).

## 1. Generating the truth table for basic logic gates
🎯 **What question does this method answer?** With n input variables, how to list all 2ⁿ combinations completely and exactly, avoiding omissions or repeats when working by hand on paper.

In [1]:
import itertools

def truth_table(fn, n_vars, names=None):
    names = names or [chr(ord('A')+i) for i in range(n_vars)]
    rows = []
    for combo in itertools.product([0,1], repeat=n_vars):
        rows.append((combo, fn(*combo)))
    header = " ".join(names) + " | Y"
    print(header); print("-"*len(header))
    for combo, y in rows:
        print(" ".join(str(c) for c in combo) + f" | {y}")
    return rows

print("=== NAND (2 inputs) ===")
truth_table(lambda a,b: int(not (a and b)), 2)
print()
print("=== XOR (2 inputs) ===")
truth_table(lambda a,b: a ^ b, 2)

=== NAND (2 inputs) ===
A B | Y
-------
0 0 | 1
0 1 | 1
1 0 | 1
1 1 | 0

=== XOR (2 inputs) ===
A B | Y
-------
0 0 | 0
0 1 | 1
1 0 | 1
1 1 | 0


[((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]

#### 📤 Real output
The NAND table gives 0 ONLY when both inputs are 1: exactly matching the original question "A two-input NAND gate will
produce a logic 0 output when both inputs are at logic 1" (Tooley Ch.5 Q3).

## 2. Simplifying Boolean expressions with sympy
🎯 **What question does this method answer?** Manual simplification easily goes wrong in which law to apply first: use sympy as an independent "referee" to confirm whether the manual result is right.

In [2]:
from sympy import symbols, simplify_logic, Or, And, Not
from sympy.logic.boolalg import to_dnf

A, B, C = symbols('A B C')

# Example from the website: A + A'.B  (module 02, generated question) -> expected to simplify to A + B
expr = Or(A, And(Not(A), B))
simplified = simplify_logic(expr)
print("Original expr :  A + A'.B")
print("Simplified (sympy):", simplified)
print("Matches the expected 'A + B'?", str(simplified).replace(' ', '') in ("Or(A,B)",) or simplified.equals(Or(A,B)))

# De Morgan: (A.B)' = A' + B'
lhs = Not(And(A, B))
rhs = Or(Not(A), Not(B))
print("\n(A.B)\' equivalent to A\'+B\' ?", simplify_logic(lhs).equals(simplify_logic(rhs)))

Original expr :  A + A'.B
Simplified (sympy): A | B
Matches the expected 'A + B'? True

(A.B)' equivalent to A'+B' ? True


#### 📤 Real output
`sympy` confirms that A + A'B simplifies correctly to A + B, and (A·B)′ is equivalent to A′+B′: exactly matching the De Morgan theorem presented in the formula block on this module's HTML page.

## 3. Practice: enter your own expression
Change `MY_EXPR` (sympy syntax: `&`=AND, `|`=OR, `~`=NOT) and re-run.

In [3]:
from sympy import sympify

MY_EXPR = "A & (B | ~A)"   # put your own expression here
expr = sympify(MY_EXPR)
print("Expression:", expr)
print("Simplified:", simplify_logic(expr))

Expression: A & (B | ~A)
Simplified: A & B


---
## 4. Floyd problem bank: Chapter 4 "Problems" (Boolean algebra, auto-solved with sympy)

Floyd Ch.4 (Boolean Algebra and Logic Simplification) has ~72 problems. Most of the pure
Boolean-algebra ones (laws, De Morgan, simplification, SOP/POS, truth tables, Karnaugh maps,
Quine-McCluskey) are **auto-solvable with `sympy.logic`** (`simplify_logic`, `to_dnf`, `to_cnf`)
instead of manual K-map drawing. Problems needing specific circuit figures (12-17, 22, 36, 45-47,
50-51) or other tools (VHDL 60-61, hardware design 62-72) are explicitly listed as **NOT solved**
rather than guessed.

⚠️ **Finding when cross-checking**: problem 8(b): the book asks to identify the Boolean law behind the equality `AAB + ABC + ABB = ABC`. Simplifying with sympy gives `AAB+ABC+ABB = AB` (not `ABC`): the two sides are NOT algebraically equivalent. This is stated explicitly in the output instead of asserting a false equality.

In [4]:
# -*- coding: utf-8 -*-
"""Auto-solve Floyd Digital Fundamentals Ch.4 'Problems' using sympy (Boolean algebra,
DeMorgan, truth tables, SOP/POS, Karnaugh-map minimization, Quine-McCluskey minterms)."""
from sympy import symbols, Or, And, Not, simplify_logic, POSform, SOPform
from sympy.logic.boolalg import to_dnf, to_cnf

A,B,C,D,E,F,G,H,I,W,X,Y,Z = symbols('A B C D E F G H I W X Y Z')

print("="*70, "\nSection 4-1 Boolean Operations and Expressions\n" + "="*70)
print("1) Expression = 0 ONLY WHEN all of A,B,C,D are 0  ->  X = A+B+C+D")
print("   (check: OR = 0 only when every variable = 0)")
print("2) Expression = 1 when ONE OR MORE of the variables A,B,C,D,E = 0  ->  X = A'+B'+C'+D'+E'")
print("3) Expression = 0 when ONE OR MORE of the variables A,B,C = 0  ->  X = ABC (AND, =0 when any variable =0)")
print("4) Evaluate:")
vals = {
  "0+0+0+0": 0, "0+0+0+1": 1, "1+1+1+1": 1,
  "1*1+0*0+1": 1, "1*0*1*0": 0, "1*0+1*0+0*1+0*1": 0,
}
for k,v in vals.items(): print(f"   {k} = {v}")
print("5) Variable values for product=1 (needs ALL variables=1, complemented ones =0) / sum=0 (needs ALL variables=0, complemented ones =1):")
print("   (a) ABC=1 when A=B=C=1")
print("   (b) A+B+C=0 when A=B=C=0")
print("   (c) A'B'C'=1 when A=B=C=0")
print("   (d) A'+B'+C'=0 when A=B=C=1")
print("   (e) A'+B+C'=0 when A=1,B=0,C=1")
print("   (f) A+B'+C=0 when A=0,B=1,C=0")
print("6) Build the truth table of X for each expression (with sympy, checked automatically):")
import itertools
def truth_table(expr, varlist, label):
    print(f"   {label} = {expr}")
    for combo in itertools.product([0,1], repeat=len(varlist)):
        subs = dict(zip(varlist, combo))
        val = int(bool(expr.subs(subs)))
        print("     ", dict(zip([str(v) for v in varlist], combo)), "-> X =", val)
truth_table(Or(A,B,C), [A,B,C], "6a) X=A+B+C")
truth_table(And(Or(A,B),C), [A,B,C], "6b) X=(A+B)C")
truth_table(And(Or(A,B),Or(B,C)), [A,B,C], "6c) X=(A+B)(B+C)")
truth_table(Or(Or(A,B), Or(And(A,B),And(B,C))), [A,B,C], "6d) X=(A+B)+(AB+BC)")
truth_table(And(Or(Not(A),B), Or(A,Not(B))), [A,B], "6e) X=(A'+B)(A+B')")

print("\n" + "="*70, "\nSection 4-2 Laws and Rules of Boolean Algebra\n" + "="*70)
print("7) Identify the Boolean law for each equality:")
print("   (a) A+AB+ABC+ABCD = ABCD+ABC+AB+A  -> COMMUTATIVE law of addition (the order of the terms is swapped)")
print("   (b) A+AB+ABC+ABCD = DCBA+CBA+BA+A  -> COMMUTATIVE law of multiplication (the order of variables in the product is swapped)")
print("   (c) AB(CD+C'D'+EF+E'F') = ABCD+ABC'D'+ABEF+ABE'F'  -> DISTRIBUTIVE law")
print("8) Identify the Boolean law/rule:")
print("   (a) AB+CD+EF = AB+CD+EF -> IDENTITY law (trivial)")
print("   (b) AAB+ABC+ABB = ABC -> rules AA=A and A.A'=0 (AAB=AB, ABB=AB -> AB+ABC+AB=ABC? check with sympy)")
lhs = simplify_logic(Or(And(A,A,B), And(A,B,C), And(A,B,B)))
rhs = simplify_logic(And(A,B,C))
print("      simplify(AAB+ABC+ABB) =", lhs, " compared with ABC =", rhs, " => match!" if lhs==rhs else " => NO match")
print("   (c) A(BC+B'C')+AC = A(BC)+AC -> absorption rule, B'C' is eliminated once AC is present")
print("   (d) AB(C+C')+AC = AB+AC -> rule C+C'=1 (complement)")
print("   (e) AB+ABC = AB -> absorption rule A+AB=A in extended form")
print("   (f) ABC+AB+ABCD = ABC+AB+D -> NEEDS REVIEW: ABC+AB=AB (absorption), if so the left side = AB+ABCD;")
print("      check with sympy whether the 'left side' and 'right side' are really equivalent in the right variables:")
lhs2 = simplify_logic(Or(And(A,B,C), And(A,B), And(A,B,C,D)))
print("      simplify(ABC+AB+ABCD) =", lhs2, "(only AB remains, differing from a lone D -> the problem may have a misprint or need an extra condition)")

print("\n" + "="*70, "\nSection 4-3 DeMorgan's Theorems (each problem verified with sympy)\n" + "="*70)
def demorgan(expr, label):
    d = to_dnf(expr, simplify=True)
    print(f"   {label}: {expr}  ->  {d}")
demorgan(Not(Or(A,B)), "9a) (A+B)'")
demorgan(Not(And(A,B)), "9b) (AB)'")
demorgan(Not(Or(A,B,C)), "9c) (A+B+C)'")
demorgan(Not(And(A,B,C)), "9d) (ABC)'")
demorgan(Not(And(A,Or(B,C))), "9e) [A(B+C)]'")
demorgan(Not(Or(And(A,B),And(C,D))), "9f) (AB+CD)'")
demorgan(Not(And(Or(A,B),Or(C,D))), "9h) [(A+B)(C+D)]'")

print("\n" + "="*70, "\nSection 4-4 Boolean Analysis / Section 4-5 Simplification (sympy simplify_logic)\n" + "="*70)
exprs_19 = {
  "19a) A(A+B)": And(A,Or(A,B)),
  "19b) A(A+AB)": And(A,Or(A,And(A,B))),
  "19c) BC+BC'": Or(And(B,C),And(B,Not(C))),
  "19d) A(A+AB) [trung 19b]": And(A,Or(A,And(A,B))),
  "19e) ABC+ABC'+A'BC": Or(And(A,B,C),And(A,B,Not(C)),And(Not(A),B,C)),
}
for k,e in exprs_19.items():
    print(f"   {k} = {e}  ->  simplified: {simplify_logic(e)}")

exprs_20 = {
  "20a) (A+B)(A+C)": And(Or(A,B),Or(A,C)),
  "20b) AB+ABC+ABCD+ABCDE": Or(And(A,B),And(A,B,C),And(A,B,C,D),And(A,B,C,D,E)),
  "20c) BC+BCD+B": Or(And(B,C),And(B,C,D),B),
  "20e) BC+(B+C)D+BC (with BC=B AND C, not to be confused with B'C')": Or(And(B,C),And(Or(B,C),D),And(B,C)),
}
for k,e in exprs_20.items():
    print(f"   {k}  ->  simplified: {simplify_logic(e)}")

print("\n" + "="*70, "\nSection 4-6 Standard SOP/POS forms\n" + "="*70)
def to_sop(expr, label):
    print(f"   {label}: {expr}  ->  standard SOP (DNF): {to_dnf(expr, simplify=True)}")
to_sop(And(Or(C,D),Or(A,D)), "23a) (C+D)(A+D)")
to_sop(And(A,Or(And(A,D),C)), "23b) A(AD+C)")
to_sop(And(Or(A,C),Or(And(C,D),And(A,C))), "23c) (A+C)(CD+AC)")

print("\n" + "="*70, "\nSection 4-7 Boolean Expressions & Truth Tables\n" + "="*70)
def sop_truth_table(minterms_expr, varlist, label):
    print(f"   {label}")
    truth_table(minterms_expr, varlist, "")
e31a = Or(And(A,B,C), And(Not(A),B,C), And(A,B,Not(C)))
sop_truth_table(e31a, [A,B,C], "31a) X = ABC + A'BC + AB C'")

print("\n" + "="*70, "\nSection 4-9/4-10 Karnaugh map minimization -- verified with sympy simplify_logic\n" + "="*70)
kmap_exprs = {
  "40a) A'B'C+A'BC+ABC": Or(And(Not(A),Not(B),C), And(Not(A),B,C), And(A,B,C)),
  "40b) AC(B+C)": And(A,C,Or(B,C)),
  "41a) A'B'C+ABC+ABC'+AB'C": Or(And(Not(A),Not(B),C),And(A,B,C),And(A,B,Not(C)),And(A,Not(B),C)),
}
for k,e in kmap_exprs.items():
    print(f"   {k}  ->  simplified (K-map == sympy simplify_logic): {simplify_logic(e)}")

print("\n" + "="*70, "\nSection 4-11 Quine-McCluskey -- list of minterms (binary)\n" + "="*70)
def minterm_binary(term_vars, varlist):
    # term_vars: list of (var, is_true)
    bits = []
    d = dict(term_vars)
    for v in varlist:
        bits.append('1' if d[v] else '0')
    return "".join(bits)
print("53) X = ABC+A'BC+AB'C+ABC'+A'B'C  -> minterms (ABC in binary):")
terms53 = [
  [(A,1),(B,1),(C,1)], [(A,0),(B,1),(C,1)], [(A,1),(B,0),(C,1)],
  [(A,1),(B,1),(C,0)], [(A,0),(B,0),(C,1)],
]
for t in terms53:
    print("   ", minterm_binary(t,[A,B,C]), "=", int(minterm_binary(t,[A,B,C]),2))
print("   -> simplified with sympy simplify_logic to compare with the manual Quine-McCluskey result:")
e53 = Or(*[And(*[v if val else Not(v) for v,val in t]) for t in terms53])
print("   simplify_logic =", simplify_logic(e53))

print("\nEND OF THE AUTO-SOLVABLE PART of Chapter 4 (the remaining problems 12-17,22,36,45-47,50-51,60-72")
print("need the specific figure/circuit from the book, not guessed to avoid making things up).")


print("\n" + "="*70, "\nSection 4-6 (continued) - problems 24-30\n" + "="*70)
to_sop(Or(And(B,C),And(D,E,Or(And(B,C),And(D,E)))), "24a) BC+DE(BC+DE)")
to_sop(And(B,C,Or(Not(C),D),Or(C,E)), "24b) BC(C'D+CE)  [note: read as BC * (C'D + CE)]")
to_sop(Or(B, And(C, Or(And(B,D), And(Or(C,D),E)))), "24c) B+C[BD+(C+D)E]")
print("25) The domain of 23(a,b,c) is the set of variables that appear: (a){A,C,D} (b){A,C,D} (c){A,C,D} -> standard SOP conversion already printed in problem 23.")
print("26) Domain of 24(a,b,c): (a){B,C,D,E} (b){B,C,D,E} (c){B,C,D,E}.")
print("27-28) Binary value of each standard SOP term -> read directly from the to_dnf result printed in problems 23,24 (each term is one clear minterm).")
def sop_to_pos(expr, label):
    print(f"   {label}: {expr}  ->  standard POS (CNF): {to_cnf(expr, simplify=True)}")
sop_to_pos(And(Or(C,D),Or(A,D)), "29a) from 23a")
sop_to_pos(And(A,Or(And(A,D),C)), "29b) from 23b")

print("\n" + "="*70, "\nSection 4-7 (continued) - problems 31-35 truth tables\n" + "="*70)
e31b = Or(And(X,Y,Z),And(Not(X),Not(Y),Not(Z)),And(Not(X),Not(Y),Z),And(X,Y,Not(Z)),And(X,Not(Y),Z))
sop_truth_table(e31b, [X,Y,Z], "31b) X = XYZ+X'Y'Z'+X'Y'Z+XYZ'+XY'Z")
e33a = Or(And(A,B),And(A,B,C),And(A,Not(C)),And(A,B,Not(C)))
sop_truth_table(e33a, [A,B,C], "33a) X = AB+ABC+AC'+ABC'  [simplified first:] " + str(simplify_logic(e33a)))
e34a = And(Or(A,B,C),Or(A,Not(B),C),Or(A,B,Not(C)))
sop_truth_table(e34a, [A,B,C], "34a) X = (A+B+C)(A+B'+C)(A+B+C')  [POS]")
e35a = And(Or(A,B),Or(A,C),Or(A,B,C))
sop_truth_table(e35a, [A,B,C], "35a) X = (A+B)(A+C)(A+B+C)  [POS]")

print("\n" + "="*70, "\nSection 4-8/4-9/4-10 (continued) - problems 42-52 with sympy instead of K-map\n" + "="*70)
e42a = Or(And(A,B),And(A,B,C),And(A,B,Not(C)))
print("42a) AB+ABC+ABC' expanded to standard SOP:", to_dnf(e42a, simplify=True), " -> simplified:", simplify_logic(e42a))
e42b = Or(A, And(B,C))
print("42b) A+BC expanded to standard SOP (domain {A,B,C}):", to_dnf(e42b, simplify=True))
e44a = Or(A, And(B,C), And(C,D))
print("44a) A+BC+CD  -> simplified with sympy (to compare with the K-map):", simplify_logic(e44a))
e44e = Or(And(Not(A),Not(B)), And(A,B), And(Not(C),Not(D)), And(C,D))
print("44e) A'B'+AB+C'D'+CD -> simplified:", simplify_logic(e44e))
print("48a) POS (A+B+C)(A+B'+C)(A+B'+C') -> simplified POS:", to_cnf(And(Or(A,B,C),Or(A,Not(B),C),Or(A,Not(B),Not(C))), simplify=True))
print("49a) POS (A+B+C+D)(A+B+C'+D')(A'+B+C+D) -> simplified:",
      to_cnf(And(Or(A,B,C,D),Or(A,B,Not(C),Not(D)),Or(Not(A),B,C,D)), simplify=True))
print("52a) Convert POS (A+B)(A+C)(A+B+C) to simplified SOP:",
      simplify_logic(And(Or(A,B),Or(A,C),Or(A,B,C))))

print("\n" + "="*70, "\nSection 4-11 (continued) - problem 54: list of 4-variable minterms\n" + "="*70)
terms54 = [
  [(A,0),(B,0),(C,0),(D,1)], [(A,0),(B,0),(C,1),(D,0)], [(A,1),(B,0),(C,1),(D,0)],
  [(A,1),(B,1),(C,0),(D,0)], [(A,1),(B,1),(C,1),(D,0)], [(A,1),(B,1),(C,1),(D,1)],
  [(A,0),(B,1),(C,0),(D,0)],
]
for t in terms54:
    b = minterm_binary(t,[A,B,C,D]); print("   ", b, "=", int(b,2))
e54 = Or(*[And(*[v if val else Not(v) for v,val in t]) for t in terms54])
print("   simplify_logic (to compare with the manual Quine-McCluskey) =", simplify_logic(e54))


Section 4-1 Boolean Operations and Expressions
1) Expression = 0 ONLY WHEN all of A,B,C,D are 0  ->  X = A+B+C+D
   (check: OR = 0 only when every variable = 0)
2) Expression = 1 when ONE OR MORE of the variables A,B,C,D,E = 0  ->  X = A'+B'+C'+D'+E'
3) Expression = 0 when ONE OR MORE of the variables A,B,C = 0  ->  X = ABC (AND, =0 when any variable =0)
4) Evaluate:
   0+0+0+0 = 0
   0+0+0+1 = 1
   1+1+1+1 = 1
   1*1+0*0+1 = 1
   1*0*1*0 = 0
   1*0+1*0+0*1+0*1 = 0
5) Variable values for product=1 (needs ALL variables=1, complemented ones =0) / sum=0 (needs ALL variables=0, complemented ones =1):
   (a) ABC=1 when A=B=C=1
   (b) A+B+C=0 when A=B=C=0
   (c) A'B'C'=1 when A=B=C=0
   (d) A'+B'+C'=0 when A=B=C=1
   (e) A'+B+C'=0 when A=1,B=0,C=1
   (f) A+B'+C=0 when A=0,B=1,C=0
6) Build the truth table of X for each expression (with sympy, checked automatically):
   6a) X=A+B+C = A | B | C
      {'A': 0, 'B': 0, 'C': 0} -> X = 0
      {'A': 0, 'B': 0, 'C': 1} -> X = 1
      {'A': 0, 'B': 1

### Note: Floyd Chapter 3 "Problems" (Logic Gates)

Chapter 3 has 55 problems, but **most require redrawing timing diagrams** from waveforms shown
in the original figures (Figure 3-76 to 3-96): fundamentally a read-the-figure-and-draw task, not
a numeric problem solvable by code without first digitizing the input waveforms from the figures.
This chapter is **NOT YET auto-solved**: would need a separate image-extraction pass (similar to
the Tooley figure workflow) to fully solve.